# Books Scraper

Scrapes all 1,000 books from [books.toscrape.com](https://books.toscrape.com) (a site built for scraping practice) across 50 paginated catalogue pages. Downloads each page's HTML locally, parses title / price / star rating with BeautifulSoup, and exports the results to `data1.csv`.

In [2]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"
HTML_DIR = "html"
NUM_PAGES = 50

os.makedirs(HTML_DIR, exist_ok=True)

## Step 1: Download each catalogue page's HTML

Downloading first and parsing separately means a network hiccup doesn't force a full re-scrape — pages already saved to disk are skipped.

In [3]:
def download_pages(base_url, num_pages, out_dir):
    for i in range(1, num_pages + 1):
        path = f"{out_dir}/page{i}.html"
        if os.path.exists(path):
            continue  # already downloaded
        try:
            response = requests.get(base_url.format(i), timeout=10)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"Failed to fetch page {i}: {e}")
            continue
        with open(path, 'w', encoding="utf-8") as f:
            f.write(response.text)

download_pages(BASE_URL, NUM_PAGES, HTML_DIR)
print("Download complete.")

Download complete.


## Step 2: Parse each saved page and extract book data

In [4]:
def parse_pages(num_pages, html_dir):
    items = []
    for i in range(1, num_pages + 1):
        path = f"{html_dir}/page{i}.html"
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "html.parser")

        for article in soup.select("article.product_pod"):
            try:
                title = article.find("h3").find("a")["title"]
                price = article.select_one("p.price_color").text.split("£")[1]
                rating = article.select_one("p.star-rating")["class"][1]
                items.append([title, price, rating])
            except (AttributeError, IndexError, KeyError) as e:
                print(f"Skipped a malformed entry on page {i}: {e}")
    return items

items = parse_pages(NUM_PAGES, HTML_DIR)
df = pd.DataFrame(items, columns=["title", "price", "rating"])
print(f"Parsed {len(df)} books.")
df.head()

Parsed 1000 books.


,title,price,rating
0,A Light in the Attic,51.77,Three
1,Tipping the Velvet,53.74,One
2,Soumission,50.10,One
3,Sharp Objects,47.82,Four
4,Sapiens: A Brief History of Humankind,54.23,Five


## Step 3: Export to CSV

In [5]:
df.to_csv("data/data1.csv", index=False)
print("Saved to data/data1.csv")

Saved to data/data1.csv
